In [ ]:
# =============================
# RUN: Bottom-100k Evaluation (single cell)
# - loads model + feat scalers
# - takes the last 100k rows from 900k_test.parquet
# - normalizes features exactly like training
# - batched tokenization & inference with GPU-safe fallback
# - metrics, leaderboard, ROC/PR, calibration, ranking metrics
# =============================

import os, time, gc, json
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix, brier_score_loss
)
from sklearn.calibration import calibration_curve

# ---------- CONFIG ----------
MODEL_DIR = "PathoPreter_Ready_V1"
PARQUET_900K = "900k_test.parquet"
BOTTOM_K = 100_000                      # <--- using bottom 100k as requested
OUT_PROBS = "100k_probs.npy"

# feature columns exactly as training
feature_cols = [
    "gnomad_af",
    "GERP++_RS_rankscore",
    "GERP_91_mammals_rankscore",
    "phyloP100way_vertebrate_rankscore",
    "phyloP470way_mammalian_rankscore",
    "phyloP17way_primate_rankscore",
    "phastCons100way_vertebrate_rankscore",
    "phastCons470way_mammalian_rankscore",
    "phastCons17way_primate_rankscore",
]

bench_cols = [
    "CADD_raw_rankscore","REVEL_rankscore","AlphaMissense_rankscore",
    "ClinPred_rankscore","PrimateAI_rankscore","MPC_rankscore",
    "BayesDel_addAF_rankscore","EVE_rankscore","ESM1b_rankscore"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_properties(0))

# ---------- helpers ----------
def map_label_str(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    if s.startswith("path"): return 1
    if s.startswith("ben"):  return 0
    return np.nan

class SequenceClassificationWithFeatures(nn.Module):
    def __init__(self, encoder, hidden_size, feature_dim, num_labels):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + feature_dim, hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels)
        )
    def forward(self, input_ids=None, attention_mask=None, features=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        last = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1)
        pooled = (last * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        features = features.to(pooled.device).to(pooled.dtype)
        logits = self.classifier(torch.cat([pooled, features], dim=1))
        return logits

# ---------- load metadata + scalers + tokenizer ----------
print("Loading model metadata and scalers...")
meta_path = os.path.join(MODEL_DIR, "model_metadata.json")
if not os.path.exists(meta_path):
    raise FileNotFoundError("Missing model metadata: " + meta_path)
with open(meta_path, "r") as f:
    meta = json.load(f)

feat_means_path = os.path.join(MODEL_DIR, "feat_means.npy")
feat_stds_path  = os.path.join(MODEL_DIR, "feat_stds.npy")
if not (os.path.exists(feat_means_path) and os.path.exists(feat_stds_path)):
    raise FileNotFoundError("Missing feat_means.npy or feat_stds.npy in model directory.")
feat_means = np.load(feat_means_path)
feat_stds  = np.load(feat_stds_path)

# sanity check feature lengths
if len(feat_means) != len(feature_cols):
    print("Warning: feat_means length != feature_cols. Using feature_cols from meta if present.")
    if "feature_cols" in meta:
        feature_cols = meta["feature_cols"]
        print("Using meta feature_cols length:", len(feature_cols))
    else:
        raise RuntimeError("feature_cols mismatch; check model metadata.")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/nucleotide-transformer-500m-human-ref", trust_remote_code=True)

# ---------- load model weights safely ----------
print("Loading encoder and model weights (safetensors)...")
base_encoder = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True)
hidden_size = base_encoder.config.hidden_size
FEATURE_DIM = len(feature_cols)
NUM_LABELS = meta.get("num_labels", 2)

model = SequenceClassificationWithFeatures(base_encoder, hidden_size, FEATURE_DIM, NUM_LABELS)
state_path = os.path.join(MODEL_DIR, "model.safetensors")
if not os.path.exists(state_path):
    raise FileNotFoundError("Missing model.safetensors in " + MODEL_DIR)
state = load_file(state_path)
state_t = {k: torch.as_tensor(v) for k, v in state.items()}
missing, unexpected = model.load_state_dict(state_t, strict=False)
print("Loaded weights (strict=False). Missing keys:", len(missing), "Unexpected keys:", len(unexpected))
model.to(DEVICE).eval()
torch.cuda.empty_cache()

# ---------- prepare bottom-100k dataframe ----------
print(f"\nLoading {PARQUET_900K} and slicing last {BOTTOM_K} rows (bottom)...")
if not os.path.exists(PARQUET_900K):
    raise FileNotFoundError("Missing parquet file: " + PARQUET_900K)
df_all = pd.read_parquet(PARQUET_900K)
print("Total rows in file:", len(df_all))
df = df_all.tail(BOTTOM_K).reset_index(drop=True)
print("Sliced rows:", len(df))
del df_all
gc.collect()

# ensure labels exist and map
df["labels"] = df["clean_label"].map(map_label_str)
valid_mask = ~df["labels"].isna()
print("Rows with labels present:", valid_mask.sum())
df = df.loc[valid_mask].reset_index(drop=True)
df["labels"] = df["labels"].astype(int)

# ---------- inference routine ----------
def run_inference_on_df(df, name="bottom100k", batch_candidates=(2048,1536,1024,768,512,256,128)):
    print(f"\nStarting inference on {name} ({len(df)} rows)")
    # prepare sequences and features
    seqs = df["raw_sequence"].astype(str).tolist()
    feats = df[feature_cols].copy()
    # impute missing with training means
    feats = feats.fillna(pd.Series(feat_means, index=feature_cols))
    feats_arr = feats.values.astype(np.float32)
    feats_norm = (feats_arr - feat_means.reshape(1,-1)) / (feat_stds.reshape(1,-1) + 1e-9)
    feats_tensor = torch.from_numpy(feats_norm)

    y_true = df["labels"].values

    probs = None
    chosen_bs = None
    for bs in batch_candidates:
        try:
            print(f"\nTrying batch size {bs} ...")
            probs_chunks = []
            t0 = time.time()
            for i in tqdm(range(0, len(seqs), bs), desc=f"Inference @{bs}"):
                j = min(len(seqs), i+bs)
                batch_seqs = seqs[i:j]
                tok = tokenizer(batch_seqs, padding="max_length", truncation=True,
                                max_length=tokenizer.model_max_length, return_tensors="pt")
                ids = tok["input_ids"].to(DEVICE)
                mask = tok["attention_mask"].to(DEVICE)
                feats_batch = feats_tensor[i:j].to(DEVICE, non_blocking=True)

                with torch.no_grad():
                    logits = model(ids, mask, feats_batch)
                    probs_batch = torch.softmax(logits, dim=1)[:, 1].cpu()
                    probs_chunks.append(probs_batch)

                # cleanup
                del ids, mask, feats_batch, tok, logits, probs_batch
                torch.cuda.empty_cache()

            probs = torch.cat(probs_chunks).numpy()
            chosen_bs = bs
            elapsed = (time.time() - t0) / 60.0
            print(f"Finished inference with batch={bs} in {elapsed:.2f} min")
            break

        except RuntimeError as e:
            msg = str(e).lower()
            print(f"RuntimeError at bs={bs}: {msg}")
            if "out of memory" in msg or "cuda" in msg:
                print(" -> OOM, clearing cache and trying next smaller batch")
                torch.cuda.empty_cache(); gc.collect(); time.sleep(2)
                continue
            else:
                raise

    if probs is None:
        raise RuntimeError("All batch sizes failed.")

    # save probs
    np.save(OUT_PROBS, probs)
    print("Saved probabilities to", OUT_PROBS)

    # metrics
    roc = roc_auc_score(y_true, probs)
    ap = average_precision_score(y_true, probs)
    print(f"\n{name} metrics: ROC-AUC={roc:.4f} | PR-AUC={ap:.4f} (batch={chosen_bs})")

    # confusion at 0.5
    y_pred05 = (probs >= 0.5).astype(int)
    cm05 = confusion_matrix(y_true, y_pred05)
    print("Confusion (thr=0.5):\n", cm05)

    # best F1 threshold via precision-recall curve
    prec, rec, thr = precision_recall_curve(y_true, probs)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    if len(thr) > 0:
        best_idx = np.nanargmax(f1[:-1])
        best_thr = thr[best_idx]
    else:
        best_thr = 0.5
    y_pred_best = (probs >= best_thr).astype(int)
    cm_best = confusion_matrix(y_true, y_pred_best)
    print(f"Best-F1 thr={best_thr:.4f} -> Confusion:\n", cm_best)

    # calibration
    brier = brier_score_loss(y_true, probs)
    prob_true, prob_pred = calibration_curve(y_true, probs, n_bins=10, strategy="uniform")
    print(f"Brier score: {brier:.4f}")

    # ranking metrics
    def recall_at_top_frac(scores, labels, frac):
        k = max(1, int(len(scores) * frac))
        idx = np.argsort(scores)[-k:]
        return labels[idx].sum() / (labels.sum() + 1e-12)

    print("Recall@Top1%:", recall_at_top_frac(probs, y_true, 0.01))
    print("Recall@Top5%:", recall_at_top_frac(probs, y_true, 0.05))
    print("Recall@Top10%:", recall_at_top_frac(probs, y_true, 0.10))

    # recall at fixed FPR=1%
    fpr_vals, tpr_vals, t_thresh = roc_curve(y_true, probs)
    valid = np.where(fpr_vals <= 0.01)[0]
    if len(valid) > 0:
        idx = valid[np.argmax(tpr_vals[valid])]
        rec_at_1pct, thr_at_1pct = tpr_vals[idx], t_thresh[idx]
    else:
        rec_at_1pct, thr_at_1pct = 0.0, None
    print(f"Recall @ FPR=1%: {rec_at_1pct:.4f} (thr={thr_at_1pct})")


Inference @128: 100%|██████████| 782/782 [4:10:15<00:00, 19.20s/it]  


Finished inference with batch=128 in 250.25 min
Saved probabilities to 100k_probs.npy

bottom-100000 metrics: ROC-AUC=0.9123 | PR-AUC=0.6204 (batch=128)
Confusion (thr=0.5):
 [[83418  9395]
 [ 1097  6090]]
Best-F1 thr=0.5832 -> Confusion:
 [[89911  2902]
 [ 2323  4864]]
Brier score: 0.1676
Recall@Top1%: 0.11172951161819951
Recall@Top5%: 0.4886600807012661
Recall@Top10%: 0.7528871573674689
Recall @ FPR=1%: 0.3568 (thr=0.6822442412376404)
